# Court Document Pipeline (Colab GPU)

Downloads court PDFs and extracts text — all on Google's servers using a free GPU.
No files need to leave your laptop. No NVIDIA API quota consumed.

### What this notebook does
1. **Clones the repo** — gets the scraper code and `valid_cases.json` (the list of known case numbers)
2. **Downloads PDFs** from the court API directly into Colab
3. **Extracts text** — PyMuPDF first (free, instant), then Qwen2-VL-7B on GPU for scanned pages
4. **Saves to Google Drive** (persistent) and/or lets you download a zip

### Team workflow
All team members should use the **same shared Google Drive folder**. The notebook automatically
skips any case whose PDFs are already in the folder, so no duplicates.

To split the work, each person sets a different `MY_WORKER` / `TOTAL_WORKERS` in the config cell
(e.g. person 1 = `0/3`, person 2 = `1/3`, person 3 = `2/3`). Each person gets a unique slice of cases.

### Before you start
- **Runtime → Change runtime type → T4 GPU**
- Have a browser tab open to get a court session ID when prompted
- If working as a team: one person creates a shared Drive folder and shares it with the others

In [1]:
!pip install -q pymupdf transformers>=4.45 accelerate bitsandbytes qwen-vl-utils Pillow

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected!\nGo to Runtime > Change runtime type > T4 GPU")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU ready: {gpu_name} ({gpu_mem:.1f} GB)")

## Setup

Clone the repo (gets the scraper code + `valid_cases.json`) and optionally
mount Google Drive so downloaded files persist between sessions.

In [ ]:
import os
from pathlib import Path

# ============================================================
# CONFIGURATION — edit these if needed
# ============================================================
REPO_URL = "https://github.com/karimiborna/litigation-outcome-pipeline.git"
BRANCH = "phase-1/scraper-and-data"

USE_DRIVE = True  # Set False to skip Drive (files won't persist)
DRIVE_DIR = "/content/drive/MyDrive/litigation-pipeline"

# TEAM WORK SPLITTING — divide cases across team members
# Each person picks a unique MY_WORKER number (0-indexed).
# Example for 3 people: person A = 0/3, person B = 1/3, person C = 2/3
# Set TOTAL_WORKERS = 1 to download everything yourself.
MY_WORKER = 0  # Your worker number (0, 1, 2, ...)
TOTAL_WORKERS = 1  # Total number of team members downloading
# ============================================================

REPO_DIR = "/content/litigation-outcome-pipeline"

if not Path(REPO_DIR).exists():
    !git config --global credential.helper ''
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

# Install the project so we can import scraper modules
!pip install -q -e {REPO_DIR}

# Set up output directories
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PDF_DIR = f"{DRIVE_DIR}/pdfs"
    OUTPUT_DIR = f"{DRIVE_DIR}/extracted"
else:
    PDF_DIR = f"{REPO_DIR}/data/raw/pdfs"
    OUTPUT_DIR = f"{REPO_DIR}/data/processed/extracted"

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nPDFs will be saved to:  {PDF_DIR}")
print(f"Text will be saved to: {OUTPUT_DIR}")

## Step 1 — Download PDFs from the court API

Reads `valid_cases.json` from the cloned repo and downloads documents for each case.
**Only useful document types are downloaded** (claims, judgments, orders, dismissals).
Procedural docs (proof of service, continuances, etc.) are skipped automatically.

**To get a session ID:**
1. Open https://webapps.sftc.org/cc/CaseCalendar.dll in your browser
2. Copy the hex string after `SessionID=` in the URL

In [ ]:
import json
import re
import time

import requests

BASE_URL = "https://webapps.sftc.org"
CASE_PATH = "/ci/CaseInfo.dll"
USER_AGENT = "MSDS603-Research-Scraper/1.0 (SF Small Claims academic study)"
DOWNLOAD_DELAY = 2.5  # seconds between requests


def ask_session_id():
    print("\n" + "=" * 50)
    print("SESSION ID NEEDED")
    print("1. Open https://webapps.sftc.org/cc/CaseCalendar.dll")
    print("2. Copy the hex string after SessionID= from the URL")
    print("=" * 50)
    sid = input("Paste SessionID: ").strip()
    return sid


def get_documents(case_num, session_id):
    url = (
        f"{BASE_URL}{CASE_PATH}/datasnap/rest/TServerMethods1/GetDocuments/{case_num}/{session_id}/"
    )
    resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    if data["result"][0] == -1:
        raise Exception("SESSION_EXPIRED")
    if data["result"][0] == 0:
        return []
    return json.loads(data["result"][1])


def sanitize(desc, max_len=40):
    return re.sub(r"[^\w\-]", "_", desc)[:max_len]


DOC_TYPE_WHITELIST = {
    "CLAIM_OF_PLAINTIFF",
    "DEFENDANT_S_CLAIM",
    "JUDGMENT",
    "ORDER",
    "DISMISSAL",
    "Notice_of_Entry_of_Judgment",
    "DECLARATION_OF_APPEARANCE",
    "STIPULATION",
    "COURT_JUDGMENT",
}


def is_doc_wanted(description):
    desc_upper = description.upper()
    return any(w.upper() in desc_upper for w in DOC_TYPE_WHITELIST)


    
# Load valid case numbers from the cloned repo
valid_cases_path = Path(REPO_DIR) / "scraper" / "state" / "valid_cases.json"
with open(valid_cases_path) as f:
    store = json.load(f)
all_valid = sorted(store.get("valid", {}).keys())
print(f"Valid cases from repo: {len(all_valid)}")

# Split across workers — each person gets every Nth case
my_cases = [c for i, c in enumerate(all_valid) if i % TOTAL_WORKERS == MY_WORKER]
print(f"Worker {MY_WORKER}/{TOTAL_WORKERS}: assigned {len(my_cases)} cases")

# Skip cases whose PDFs are already in the shared Drive folder
to_download = []
for case_num in my_cases:
    existing = list(Path(PDF_DIR).glob(f"{case_num}_*.pdf"))
    if not existing:
        to_download.append(case_num)

print(f"Already in Drive (skipped): {len(my_cases) - len(to_download)}")
print(f"To download: {len(to_download)}")

if to_download:
    session_id = ask_session_id()
    http = requests.Session()
    http.headers["User-Agent"] = USER_AGENT
    total_pdfs = 0

    for i, case_num in enumerate(to_download):
        # Re-check in case a teammate downloaded this while we were running
        if list(Path(PDF_DIR).glob(f"{case_num}_*.pdf")):
            continue

        try:
            time.sleep(DOWNLOAD_DELAY)
            docs = get_documents(case_num, session_id)
        except Exception as e:
            if "SESSION_EXPIRED" in str(e):
                session_id = ask_session_id()
                time.sleep(DOWNLOAD_DELAY)
                docs = get_documents(case_num, session_id)
            else:
                print(f"  Error for {case_num}: {e}")
                continue

        for doc in docs:
            doc_url = doc.get("URL", "")
            desc = doc.get("DESCRIPTION", "doc")
            if not doc_url or not is_doc_wanted(desc):
                continue

            pdf_path = Path(PDF_DIR) / f"{case_num}_{sanitize(desc)}.pdf"
            if pdf_path.exists():
                continue

            time.sleep(DOWNLOAD_DELAY)
            try:
                resp = http.get(doc_url, timeout=60, stream=True)
                if "pdf" in resp.headers.get("content-type", "").lower():
                    with open(pdf_path, "wb") as f:
                        for chunk in resp.iter_content(8192):
                            f.write(chunk)
                    total_pdfs += 1
            except Exception as e:
                print(f"  Download failed: {e}")

        if (i + 1) % 10 == 0:
            print(f"  Progress: {i + 1}/{len(to_download)} cases, {total_pdfs} PDFs")

    print(f"\nDownloaded {total_pdfs} PDFs")
else:
    print("All PDFs already downloaded.")

# List what we have
all_pdfs = sorted(f.name for f in Path(PDF_DIR).glob("*.pdf"))
print(f"\nTotal PDFs on disk: {len(all_pdfs)}")

## Step 2 — PyMuPDF (free text extraction)

Pulls selectable text from each PDF instantly. No GPU needed.
Only PDFs that are scanned images (< 100 chars of text) move to the GPU step.

In [ ]:
import fitz  # PyMuPDF

MIN_TEXT_LENGTH = 100


def extract_with_pymupdf(pdf_path):
    doc = fitz.open(str(pdf_path))
    pages = [page.get_text() for page in doc]
    doc.close()
    return "\n\n".join(pages).strip()


all_pdfs = sorted(Path(PDF_DIR).glob("*.pdf"))

# Only extract whitelisted document types — skip procedural PDFs
pdfs = [p for p in all_pdfs if is_doc_wanted(p.stem.split("_", 1)[-1] if "_" in p.stem else p.stem)]
skipped_types = len(all_pdfs) - len(pdfs)

already_done = []
pymupdf_extracted = []
needs_gpu = []

for pdf_path in pdfs:
    txt_path = Path(OUTPUT_DIR) / f"{pdf_path.stem}.txt"

    if txt_path.exists():
        already_done.append(pdf_path)
        continue

    text = extract_with_pymupdf(pdf_path)
    if len(text) > MIN_TEXT_LENGTH:
        txt_path.write_text(text, encoding="utf-8")
        pymupdf_extracted.append(pdf_path)
    else:
        needs_gpu.append(pdf_path)

print(f"Skipped (not in whitelist):  {skipped_types}")
print(f"Already extracted (skipped): {len(already_done)}")
print(f"PyMuPDF extracted this run:  {len(pymupdf_extracted)}")
print(f"Need GPU vision model:       {len(needs_gpu)}")

## Step 3 — Load Vision Model

Loads **Qwen2-VL-7B-Instruct** with 4-bit quantization (~4 GB VRAM).
First run downloads ~4 GB of weights (cached for subsequent runs).

Skipped entirely if all PDFs had selectable text.

In [6]:
model = None
processor = None

if needs_gpu:
    from transformers import (
        AutoProcessor,
        BitsAndBytesConfig,
        Qwen2VLForConditionalGeneration,
    )

    MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    print(f"Loading {MODEL_ID} (4-bit quantized)...")
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)

    used = torch.cuda.memory_allocated() / 1e9
    print(f"Model loaded. GPU memory used: {used:.1f} GB")
else:
    print("All PDFs had selectable text — no model needed.")

## Step 4 — Extract scanned PDFs with GPU

Renders each page to an image and sends it through the vision model.
Uses a **targeted prompt** for claim documents (SC-100 forms) that skips bilingual
boilerplate and focuses on the filled-in case data. Other document types get the
generic extraction prompt. Pages with no case data are skipped entirely.

In [ ]:
import io

from PIL import Image
from qwen_vl_utils import process_vision_info

GENERIC_PROMPT = (
    "This is a page from a San Francisco small claims court document. "
    "Extract all text exactly as it appears. Include party names, dates, "
    "claim amounts, rulings, and any other information on the page. "
    "Output plain text only."
)

CLAIM_PROMPT = (
    "This is a page from a California SC-100 small claims complaint form. "
    "SKIP all boilerplate instructions, bilingual notices, and form guidance. "
    "Extract ONLY the filled-in case-specific information:\n"
    "- Plaintiff name(s), address, phone\n"
    "- Defendant name(s), address, phone\n"
    "- Trial date and department\n"
    "- Dollar amount claimed and basis for the amount\n"
    "- Description of what happened (the plaintiff's narrative)\n"
    "- Any evidence, witnesses, or documents referenced\n"
    "- Whether plaintiff tried to resolve before suing\n\n"
    "If a page is only boilerplate instructions with no filled-in data, "
    "respond with: [NO CASE DATA ON THIS PAGE]\n"
    "Output plain text only."
)
DPI = 150


def get_prompt_for_doc(filename):
    name_upper = filename.upper()
    if "CLAIM_OF_PLAINTIFF" in name_upper or "DEFENDANT_S_CLAIM" in name_upper:
        return CLAIM_PROMPT
    return GENERIC_PROMPT


def page_to_pil(page, dpi=DPI):
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    return Image.open(io.BytesIO(pix.tobytes("png")))


def extract_page(pil_image, prompt):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=2048, do_sample=False)

    generated = output_ids[0, inputs.input_ids.shape[1] :]
    return processor.decode(generated, skip_special_tokens=True)


if needs_gpu and model is not None:
    t0 = time.time()
    for i, pdf_path in enumerate(needs_gpu):
        txt_path = Path(OUTPUT_DIR) / f"{pdf_path.stem}.txt"

        if txt_path.exists():
            continue

        prompt = get_prompt_for_doc(pdf_path.name)
        doc = fitz.open(str(pdf_path))
        pages_text = []

        for page_num, page in enumerate(doc, start=1):
            print(
                f"  [{i + 1}/{len(needs_gpu)}] {pdf_path.name} - page {page_num}/{len(doc)}",
                flush=True,
            )
            img = page_to_pil(page)
            text = extract_page(img, prompt)
            if "[NO CASE DATA ON THIS PAGE]" not in text:
                pages_text.append(text)

        doc.close()
        torch.cuda.empty_cache()

        full_text = "\n\n--- PAGE BREAK ---\n\n".join(pages_text)
        txt_path.write_text(full_text, encoding="utf-8")
        print(f"  Saved: {txt_path.name} ({len(full_text):,} chars)")

    elapsed = time.time() - t0
    print(f"\nGPU extraction done: {len(needs_gpu)} PDFs in {elapsed:.0f}s")
else:
    print("No scanned PDFs to process.")

## Step 5 — Extract labels with GPT-4o-mini

Sends outcome documents (judgments, orders, dismissals) to GPT-4o-mini to extract
structured labels: win/loss/dismissed, amount awarded, whether defendant appeared, etc.

Requires an OpenAI API key (set `LLM_API_KEY` below). Costs ~$0.001 per case.

In [9]:
import sys

sys.path.insert(0, REPO_DIR)


LLM_API_KEY = input("OpenAI API key (leave blank to skip label extraction): ").strip()

if LLM_API_KEY:
    from features.config import FeaturesConfig
    from features.labels import LabelExtractor, gather_outcome_text

    label_config = FeaturesConfig(
        llm_api_key=LLM_API_KEY,
        llm_model="gpt-4o-mini",
        cache_dir=f"{OUTPUT_DIR}/../labels_cache",
    )
    extractor = LabelExtractor(config=label_config)

    txt_dir = Path(OUTPUT_DIR)
    case_numbers = sorted(
        {f.name.split("_")[0] for f in txt_dir.glob("*.txt") if f.name.startswith("CSM")}
    )
    print(f"Found {len(case_numbers)} cases to label")

    all_labels = {}
    for i, case_num in enumerate(case_numbers):
        outcome_text = gather_outcome_text(case_num, txt_dir)
        if not outcome_text:
            continue
        labels = extractor.extract_labels(case_num, txt_dir)
        if labels:
            all_labels[case_num] = labels.model_dump()
            print(
                f"  [{i + 1}/{len(case_numbers)}] {case_num}: {labels.outcome} "
                f"(${labels.total_awarded or 0:.2f})"
            )

    labels_path = Path(OUTPUT_DIR) / "../labels.json"
    labels_path.write_text(json.dumps(all_labels, indent=2), encoding="utf-8")
    print(f"\nLabels saved: {labels_path} ({len(all_labels)} cases)")
else:
    print("Skipping label extraction (no API key).")


Searching for extracted text in:
  /content/drive/MyDrive/litigation-pipeline/extracted — 2085 txt files
  /content/drive/MyDrive/litigation-pipeline/extracted_nvidia — 656 txt files

LABEL EXTRACTION SUMMARY
Total cases with text:       1033
Worker 0/1:                    assigned 1033 cases
Already labeled (skipped):   370
To label this run:           663

  [1/663] CSM25870003 — SKIPPED (no outcome documents found)
  [2/663] CSM25870027 — SKIPPED (no outcome documents found)
  [3/663] CSM25870039 — SKIPPED (no outcome documents found)
  [4/663] CSM25870373 — 2,934 chars of outcome text (from extracted)
    -> dismissed | awarded: $0.00 | defendant appeared: no
       The court ordered dismissal of the entire action without prejudice.
  [5/663] CSM25870374 — 7,058 chars of outcome text (from extracted)
    -> plaintiff_win | awarded: $10,458.00 | defendant appeared: unknown
       The court ruled in favor of the plaintiff, Andrew Rodgers, awarding him $10,203 in principal and $255 i

## Step 6 — Extract features with GPT-4o-mini

Builds a `ProcessedCase` per case (excluding label docs to prevent leakage) and runs `FeatureExtractor` to produce the v2 existence-based feature vectors. Cache lands in Drive so cases survive runtime restarts.

Reuses the OpenAI API key entered in Step 5; prompts again if you skipped that step. Costs ~\$0.001 per case with `gpt-4o-mini`.

In [ ]:
import sys

sys.path.insert(0, REPO_DIR)

# Reuse the key from Step 5 if available; re-prompt if missing.
try:
    _existing_key = LLM_API_KEY  # noqa: F821
except NameError:
    _existing_key = ""

if not _existing_key:
    LLM_API_KEY = input("OpenAI API key (leave blank to skip feature extraction): ").strip()

if LLM_API_KEY:
    from datetime import date

    from data.schemas.case import ExtractedText, ProcessedCase
    from features.config import FeaturesConfig
    from features.extraction import FeatureExtractor
    from features.labels import LABEL_DOC_KEYWORDS

    def _is_label_txt(name: str) -> bool:
        upper = name.upper()
        return any(kw.upper() in upper for kw in LABEL_DOC_KEYWORDS)

    def build_processed_case(case_number, txt_dir):
        all_txts = sorted(txt_dir.glob(f"{case_number}_*.txt"))
        feature_txts = [p for p in all_txts if not _is_label_txt(p.name)]
        if not feature_txts:
            return None

        has_defendant_claim = any("DEFENDANT_S_CLAIM" in p.name.upper() for p in all_txts)
        user_side = "defendant" if has_defendant_claim else "plaintiff"

        document_texts = []
        text_parts = []
        for p in feature_txts:
            text = p.read_text(encoding="utf-8").strip()
            if not text:
                continue
            document_texts.append(
                ExtractedText(
                    case_number=case_number,
                    document_filename=p.stem,
                    pages=[text],
                )
            )
            text_parts.append(text)

        if not text_parts:
            return None

        return ProcessedCase(
            case_number=case_number,
            case_title="",
            cause_of_action=None,
            filing_date=date(1900, 1, 1),
            full_text="\n\n---\n\n".join(text_parts),
            document_texts=document_texts,
            claim_amount=None,
            has_attorney_plaintiff=None,
            has_attorney_defendant=None,
            user_side=user_side,
        )

    feature_config = FeaturesConfig(
        llm_api_key=LLM_API_KEY,
        llm_model="gpt-4o-mini",
        cache_dir=f"{DRIVE_DIR}/features_cache",
    )
    feature_extractor = FeatureExtractor(config=feature_config)

    txt_dir = Path(OUTPUT_DIR)
    case_numbers = sorted(
        {f.name.split("_")[0] for f in txt_dir.glob("*.txt") if f.name.startswith("CSM")}
    )

    # Deterministic worker sharding: stride over sorted case numbers.
    if TOTAL_WORKERS > 1:
        case_numbers = case_numbers[MY_WORKER::TOTAL_WORKERS]

    print(f"Found {len(case_numbers)} cases in this worker's slice")

    processed_cases = []
    for case_num in case_numbers:
        pc = build_processed_case(case_num, txt_dir)
        if pc is not None:
            processed_cases.append(pc)

    print(f"Built {len(processed_cases)} ProcessedCase objects; running extraction...")

    feature_vectors = await feature_extractor.extract_batch(processed_cases)

    print(f"\nFeature extraction complete: {len(feature_vectors)} / {len(processed_cases)} cases")
    print(f"Cache dir: {feature_config.cache_dir}")
else:
    print("Skipping feature extraction (no API key).")

## Results

Summary of all extracted text and labels. Downloads everything as a zip.

In [10]:
import shutil

txt_files = sorted(Path(OUTPUT_DIR).glob("*.txt"))
total_chars = sum(f.stat().st_size for f in txt_files)

print(f"Total text files: {len(txt_files)}")
print(f"Total size:       {total_chars:,} characters ({total_chars / 1024:.0f} KB)")
print()

for f in txt_files[:20]:
    print(f"  {f.name}  ({f.stat().st_size:,} chars)")
if len(txt_files) > 20:
    print(f"  ... and {len(txt_files) - 20} more")

labels_path = Path(OUTPUT_DIR) / "../labels.json"
if labels_path.exists():
    labels = json.loads(labels_path.read_text())
    outcomes = {}
    for v in labels.values():
        o = v.get("outcome", "unknown") or "unknown"
        outcomes[o] = outcomes.get(o, 0) + 1
    print(f"\nLabels: {len(labels)} cases")
    for outcome, count in sorted(outcomes.items()):
        print(f"  {outcome}: {count}")

if USE_DRIVE:
    print(f"\nFiles are saved to Google Drive at: {DRIVE_DIR}")
    print("They will persist even if this runtime disconnects.")

zip_path = "/content/extracted_texts"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"\nZip: {zip_path}.zip ({Path(zip_path + '.zip').stat().st_size / 1024:.0f} KB)")

from google.colab import files

files.download(f"{zip_path}.zip")

Total text files: 2085
Total size:       8,895,697 characters (8687 KB)

  CSM25870000_CLAIM_OF_PLAINTIFF.txt  (22,802 chars)
  CSM25870000_DECLARATION.txt  (1,260 chars)
  CSM25870000_DECLARATION_OF_APPEARANCE_ON_BEHALF_OF_C.txt  (2,922 chars)
  CSM25870000_ORDER.txt  (1,092 chars)
  CSM25870000_Small_Claims_-_Continue_Order.txt  (1,196 chars)
  CSM25870000_Small_Claims_Order_of_Dismissal_of_Entir.txt  (1,174 chars)
  CSM25870001_CLAIM_OF_PLAINTIFF.txt  (19,889 chars)
  CSM25870001_GENERIC_CIVIL_FILING__NO_FEE_.txt  (10,455 chars)
  CSM25870001_JUDGMENT_ON_PLAINTIFF_S_CLAIM.txt  (1,471 chars)
  CSM25870001_Notice_of_Entry_of_Judgment.txt  (3,100 chars)
  CSM25870001_ORDER.txt  (2,439 chars)
  CSM25870001_PROOF_OF_SERVICE_ON_CLAIM.txt  (5,297 chars)
  CSM25870001_WRIT_OF_EXECUTION.txt  (3,863 chars)
  CSM25870002_CLAIM_OF_PLAINTIFF.txt  (23,881 chars)
  CSM25870002_DECLARATION.txt  (967 chars)
  CSM25870002_DECLARATION_OF_DUE_DILIGENCE_IN_ATTEMPTI.txt  (4,848 chars)
  CSM25870002_DISMI

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>